# Linearized Fisher Updates for Continual Learning

This work studies continual adaptation of deep networks under constrained compute: small batches, limited accelerator memory, and a deliberately small trainable parameter subspace such as a Low Rank Adapter (LoRA) [1]. Elastic Weight Consolidation (EWC) [3] compresses earlier observations into a local quadratic approximation to their log likelihood. At time $t$, we represent that approximation by an anchor and a precision-like information summary,

$$ (\theta_t, \Lambda_t), \qquad \Lambda_t := n_t \widehat{\mathcal I}_t. $$

This pair resembles a Gaussian sufficient statistic, but it is only a local approximation in a general deep network. Its usefulness depends on keeping $\widehat{\mathcal I}_t$ coherent as the model moves through parameter space. A moving average incorporates new curvature observations but lags behind a changing parameter. The proposal here is to reduce that first-order lag with a **linearized Fisher update (LFU)**.

## Definitions and two coupled processes

Let $p_\theta(x)$ be a regular parametric model in a fixed parameter chart $\theta \in \Theta \subseteq \mathbb R^p$. Define

- the log likelihood $\ell(x;\theta) := \log p_\theta(x)$,
- the score $s(x;\theta) := \nabla_\theta \ell(x;\theta)$,
- the Fisher information matrix (FIM) $\mathcal I(\theta) := \mathbb E_\theta[s s^T]$, and
- a parameter displacement $u_t := \theta_{t+1}-\theta_t$.

For a negative log likelihood $L=-\ell$, the loss gradient is $g:=\nabla_\theta L=-s$. For a more general training loss, $g$ defines an empirical-Fisher or pseudo-score construction rather than the model Fisher; the distinction should be made explicit in each experiment.

Two related processes must not be conflated:

1. **Original learning process.** The optimizer, policy update, or adaptation rule uses incoming data to choose $u_t$ and moves $\theta_t$ to $\theta_{t+1}$. This notebook does not prescribe how $u_t$ is chosen; it assumes that the resulting steps are small enough for a local expansion to be useful.
2. **Auxiliary Fisher process.** Given the path produced by the original process, this process estimates the Fisher field along that path and updates $\widehat{\mathcal I}_t$ or $\Lambda_t$. It does not choose the destination. It consumes $u_t$ and asks how the information summary should change because of that move.

In short, the original process decides where the model moves; the auxiliary Fisher process tries to keep the compressed information summary coherent after the move. The auxiliary process is therefore well defined without inventing a Gaussian score family or assuming that it can be placed in canonical coordinates.

## Math sketch: the full Fisher derivative

For any sufficiently regular scalar or matrix-valued function $a(X,\theta)$,

$$ \partial_k \mathbb E_\theta[a(X,\theta)] = \mathbb E_\theta[\partial_k a(X,\theta) + a(X,\theta)s_k(X;\theta)]. $$

Applying this identity to $a=s_i s_j$ gives

$$ \partial_k \mathcal I_{ij}(\theta) = \mathbb E_\theta[(\partial_k s_i)s_j + s_i(\partial_k s_j) + s_i s_j s_k]. $$

Define the Amari-Chentsov tensor [2]

$$ C_{ijk}(\theta) := \mathbb E_\theta[s_i s_j s_k], $$

and define the **residual tensor**

$$ R_{ijk}(\theta) := \mathbb E_\theta[(\partial_k s_i)s_j + s_i(\partial_k s_j)]. $$

Using Einstein summation and $(T:u)_{ij}:=T_{ijk}u^k$, the directional derivative of the Fisher matrix is

$$ D\mathcal I_\theta[u] = (C_\theta+R_\theta):u. $$

The LFU is the first-order Taylor approximation

$$ \boxed{\mathcal I(\theta+u) = \mathcal I(\theta) + (C_\theta+R_\theta):u + O(\|u\|^2).} $$

For a canonical exponential family in its natural coordinates, $\partial_k s_i$ is deterministic and $\mathbb E_\theta[s_j]=0$, so $R_{ijk}=0$ and the Amari-Chentsov term is enough. A deep network in its ordinary weight coordinates is not generally in this case. Gaussianizing an estimator does not supply the unknown map from a model displacement $u$ to a canonical precision displacement, so it does not remove $R$.

The name residual tensor is operational. Strictly, the components $R_{ijk}$ are coordinate dependent under nonlinear reparameterization: they are a combination of connection-coefficient terms, whereas $C_{ijk}$ is an intrinsic tensor. Accordingly, an LFU is a coordinate-local Taylor update, not parallel transport. This is acceptable for the intended implementation, which remains in one fixed deep-network parameter chart or one fixed low-dimensional adapter chart.

## A matrix-free LFU estimator

Let

$$ h_u(X;\theta) := \nabla_\theta^2 \ell(X;\theta)u. $$

Contracting before taking expectations avoids materializing either rank-three array:

$$ D\mathcal I_\theta[u] = \mathbb E_\theta\left[h_u s^T + s h_u^T + (u^T s)ss^T\right]. $$

With the loss convention $L=-\ell$, $g=\nabla L$, and $H_u=\nabla^2 L\,u$, the same identity is

$$ \boxed{D\mathcal I_\theta[u] = \mathbb E_\theta\left[H_u g^T + gH_u^T - (u^T g)gg^T\right].} $$

Thus a sample LFU needs a per-sample gradient, one Hessian-vector product (HVP), and the scalar $u^Tg$. Reverse-mode autodiff computes $H_u$ by differentiating $g^Tu$; no dense Hessian is formed [4]. Moreover, with

$$ U=[g,H_u], \qquad B=\begin{bmatrix}-(u^Tg)&1\\1&0\end{bmatrix}, $$

the sample correction is $UBU^T$ and has rank at most two. An average of these corrections can be accumulated, sketched, truncated, or applied as a linear operator without storing $C$, $R$, or a dense Hessian. The correction is symmetric but need not be positive semidefinite; damping or projection may be needed after a finite first-order step.

## Computational regime

The proposal targets a specific regime:

- Only a manageable parameter subspace is adapted. Full-model LFUs are not assumed practical for large networks.
- Data arrive in small batches, possibly one observation at a time. Per-sample gradients must be retained or vectorized because an outer product of the batch-mean gradient is not the mean of per-sample outer products.
- HVPs are computed by autodiff. Relative to ordinary training, this adds roughly one backward-like pass and retains a higher-order graph, increasing both time and activation memory, but it does not store a $p\times p$ Hessian.
- Fisher summaries and LFU corrections use diagonal, block-diagonal, Kronecker-factored, low-rank, or sketched representations when a dense $p\times p$ matrix is too large.
- The model moves continuously enough that the omitted $O(\|u_t\|^2)$ term is controlled. Large steps should trigger re-estimation, subdivision into smaller LFUs, or rejection of the linear approximation.

Large sample limits may justify separate Gaussian or SDE models of the original learning process, but they are not needed for the LFU identity itself.

## EWC as local information compression

Suppose old observations are summarized at $\theta_t$ by $\Lambda_t=n_t\widehat{\mathcal I}_t$. For a candidate parameter $\vartheta$, their log likelihood is approximated by

$$ \log p(X_{\mathrm{old}};\vartheta) \approx \mathrm{const} - \frac12(\vartheta-\theta_t)^T\Lambda_t(\vartheta-\theta_t). $$

If the new-data likelihood is also approximated near its local MLE $\widehat\theta_{\mathrm{new}}$ with precision $\Lambda_{\mathrm{new}}$, then the EWC-regularized local MLE has the Gaussian product form

$$ \boxed{\widehat\theta_{t+1}=(\Lambda_t+\Lambda_{\mathrm{new}})^{-1}(\Lambda_t\theta_t+\Lambda_{\mathrm{new}}\widehat\theta_{\mathrm{new}}).} $$

The combined precision and natural parameter are

$$ \Lambda_{t+1}=\Lambda_t+\Lambda_{\mathrm{new}}, \qquad \eta_{t+1}=\Lambda_t\theta_t+\Lambda_{\mathrm{new}}\widehat\theta_{\mathrm{new}}. $$

These equations are exact for the two quadratic approximations. They reveal the most economical representation of accumulated quadratic evidence: Gaussian natural parameters $(\Lambda,\eta)$ add. Re-anchoring the same fixed quadratic does not require an LFU and does not change $\Lambda$; its center is recovered by solving $\Lambda\theta=\eta$. In a low-rank or singular trainable subspace, that solve requires damping, a pseudoinverse, or a structured solver.

This exposes an important target distinction. An LFU predicts the **model Fisher field** under the moving model distribution,

$$ \widehat{\mathcal I}_{t\to t+1}^{\mathrm{LFU}} := \widehat{\mathcal I}_t + \widehat{D\mathcal I_{\theta_t}[u_t]}. $$

It does not exactly update the observed information of a fixed old dataset. The latter is $-\nabla^2_\theta\log p(X_{\mathrm{old}};\theta)$, whose change at fixed $X_{\mathrm{old}}$ involves third derivatives of that old-data likelihood rather than $D\mathcal I_\theta[u]$. LFUs are therefore coherent for maintaining an estimate of current local model Fisher, which can shape future EWC factors as the agent traverses the manifold. Using an LFU to evolve the curvature of an already-compressed old-data factor is a further adaptive-EWC approximation and must be tested as such.

## Experimental considerations

LFUs trade moving-average lag for derivative-estimation error. The Amari-Chentsov contribution is a high-variance third score moment, while the residual contribution requires noisy HVPs. Although each contracted sample update is low rank, many updates can accumulate rank and may require truncation. The first-order approximation can also lose positive semidefiniteness or fail when $u_t$ is too large.

The first MNIST experiment should compare four Fisher estimators against frequent re-estimation at the current parameter:

1. an exponential moving average (EMA) baseline,
2. an Amari-Chentsov-only update $C:u_t$,
3. the full LFU $(C+R):u_t$, and
4. periodic fresh Fisher estimates as a higher-compute reference.

Useful ablations include HVP frequency, batch size down to one, adapter dimension, low-rank budget, damping, step size, and subdivision of large steps. Evaluation should measure Fisher approximation error, retained-task performance, adaptation to the changing task, wall-clock time, and peak memory. The central diagnostic is whether the residual tensor materially improves prediction of $\mathcal I(\theta_t+u_t)$ over both EMA and the Amari-Chentsov-only correction.

A second experiment should test whether the same conclusions survive structured or low-rank summaries at a scale closer to the intended edge-robotics setting. A final robotics demonstration can then test whether better-maintained EWC summaries improve continual adaptation of a vision-language model without replaying the original large dataset.

## A model of the original learning process

The LFU construction deliberately leaves $u_t$ unspecified. For numerical experiments, one may separately model the original process as tracking a slowly changing data-generating distribution. Let $P_t$ denote the environment at time $t$, let incoming observations satisfy $X_t\sim P_t$, and let an adaptation rule $A_t$ produce

$$ u_t=A_t(\theta_t,X_t,\text{optimizer state}), \qquad \theta_{t+1}=\theta_t+u_t. $$

This separates three objects: environmental change in $P_t$, the learning rule that determines $u_t$, and the auxiliary Fisher process driven by that realized $u_t$. Model correctness, independence, and slow change can be imposed as experiment-specific assumptions rather than being built into the definition of LFU.

For the earlier mixture interpretation of EWC, introduce an observed Bernoulli label $M_i$ with $\mathbb P(M_i=1)=\pi$, where $M_i=0$ marks old-task observations and $M_i=1$ marks new-task observations. Expanding the old-data likelihood around its MLE $\theta_t$ gives

$$ \frac1n\log p(X;\vartheta) \approx \frac1n\log p(X^{t+1};\vartheta)-\frac{1-\pi}{2}(\vartheta-\theta_t)^T\mathcal I(\theta_t)(\vartheta-\theta_t)+\mathrm{const}. $$

This recovers the EWC penalty as a local frequentist approximation. It does not make $(\theta_t,n_t\widehat{\mathcal I}_t)$ a globally sufficient statistic, and its accuracy must be checked as the anchor moves.

## Optional diffusion approximation

A diffusion model can be useful for studying the original process, but it is an additional asymptotic model rather than a consequence of LFU. For a triangular array with many observations per small update, suppose the conditional increment satisfies

$$ \theta_{k+1,n}-\theta_{k,n}\approx \frac{\pi b(\theta_{k,n})}{n}+\sqrt{\frac{\pi}{n}}\,\mathcal I^{-1/2}(\theta_{k,n})\xi_k, \qquad \xi_k\sim_{iid}\mathcal N(0,I_p). $$

Under the usual regularity, tightness, and Lipschitz assumptions, scaling $k=\lfloor nt\rfloor$ suggests

$$ d\Theta_t=\pi b(\Theta_t)dt+\sqrt{\pi}\,\mathcal I^{-1/2}(\Theta_t)dW_t. $$

Here $b$ and $\pi$ describe how the original process moves. The auxiliary process estimates or predicts the Fisher term along the resulting path. Experiments should not use the diffusion approximation as evidence that the residual tensor vanishes.

## Why MNIST is comparable to continual reinforcement learning

Both settings can be represented as adaptation to a slowly changing stream. In reinforcement learning, policy updates change the state-action distribution observed by the agent. In the proposed MNIST experiment, the frequency of digit 9 increases gradually after an initial fit on digits 0 through 8. In both cases, the original process follows an optimizer-dependent path $\theta_t$, while the auxiliary process tries to maintain curvature information along that path.

The comparison is intentionally limited: MNIST does not reproduce temporal credit assignment, policy-dependent sampling, or nonstationary transition dynamics. It isolates the narrower question of whether LFUs improve compressed Fisher tracking under controlled distribution shift.


## Single-observation batches

Single-observation updates reduce activation memory and fit the intended online setting, but they do not remove the need for per-sample derivatives. The useful scheduling principle is that each observation corrects the auxiliary Fisher process for the preceding parameter move, then helps the original learning process choose the next move.

### Directional LFU estimator

Suppose the preceding move was $u_{t-1}:=\theta_t-\theta_{t-1}$. After arriving at $\theta_t$, draw $X_t\sim p_{\theta_t}$ and evaluate

$$ s_t:=\nabla_\theta\ell(X_t;\theta_t), \qquad h_t:=\nabla_\theta^2\ell(X_t;\theta_t)u_{t-1}. $$

The lagged single-sample LFU contribution is

$$ \widehat\Delta_t^{\mathrm{lag}}=h_t s_t^T+s_t h_t^T+(u_{t-1}^Ts_t)s_t s_t^T. $$

For a negative log likelihood, let $g_t:=\nabla_\theta L(X_t;\theta_t)$ and $H_t:=\nabla_\theta^2L(X_t;\theta_t)u_{t-1}$. Then

$$ \widehat\Delta_t^{\mathrm{lag}}=H_tg_t^T+g_tH_t^T-(u_{t-1}^Tg_t)g_tg_t^T. $$

### Online LFU recursion

First use the lagged LFU to predict the current Fisher, then blend that prediction with direct curvature evidence at $\theta_t$:

$$ \widetilde{\mathcal I}_t=\widehat{\mathcal I}_{t-1}+\widehat\Delta_t^{\mathrm{lag}}, $$

$$ \widehat{\mathcal I}_t=(1-\alpha_t)\widetilde{\mathcal I}_t+\alpha_t Z_t, \qquad Z_t:=s_ts_t^T. $$

The same observation $X_t$ may then help the original learning process choose $u_t$, after which $\theta_{t+1}=\theta_t+u_t$. Thus $X_t$ corrects the Fisher estimate for the past direction $u_{t-1}$ and creates the future direction $u_t$. The gain $\alpha_t$ controls auxiliary-process memory and is distinct from the optimizer learning rate or any control variable used by the original process.

### Lagged directions and EMA error control

Let $\mathcal F_{t-1}$ contain the history after the move to $\theta_t$ but before drawing $X_t$. Then $\theta_t$ and $u_{t-1}$ are $\mathcal F_{t-1}$-measurable. Under conditionally on-model sampling,

$$ \mathbb E\left[\widehat\Delta_t^{\mathrm{lag}}\mid\mathcal F_{t-1}\right]=D\mathcal I_{\theta_t}[u_{t-1}]. $$

This predictability is what removes the same-sample coupling bias. In contrast, pairing $X_t$ with a direction $u_t(X_t)$ generally makes the sample LFU conditionally biased; merely computing the LFU after the optimizer step does not change their shared randomness.

The lagged estimator evaluates the derivative at the endpoint of the preceding move. If $D\mathcal I$ is locally Lipschitz with constant $L$, then

$$ \mathcal I(\theta_t)=\mathcal I(\theta_{t-1})+D\mathcal I_{\theta_t}[u_{t-1}]+r_t, \qquad \|r_t\|\leq\frac{L}{2}\|u_{t-1}\|^2. $$

Define the Fisher estimation error $e_t:=\widehat{\mathcal I}_t-\mathcal I(\theta_t)$, LFU noise $\varepsilon_t:=\widehat\Delta_t^{\mathrm{lag}}-D\mathcal I_{\theta_t}[u_{t-1}]$, and direct-observation noise $\zeta_t:=Z_t-\mathcal I(\theta_t)$. The EMA recursion gives

$$ e_t=(1-\alpha_t)(e_{t-1}+\varepsilon_t-r_t)+\alpha_t\zeta_t. $$

EMA therefore prevents old errors and second-order remainders from accumulating as an uncontracted sum. For constant gain $\alpha\in(0,1]$ and uniformly bounded remainder,

$$ \|\mathbb E[e_t]\|\leq(1-\alpha)^t\|\mathbb E[e_0]\|+\frac{1-\alpha}{\alpha}\sup_j\|r_j\|. $$

For step size $\|u_t\|=O(\eta)$, the persistent truncation contribution is consequently $O((1-\alpha)\eta^2/\alpha)$ rather than an indefinitely growing $O(t\eta^2)$ drift. EMA controls this error but does not erase it. The gain also mediates a variance tradeoff: $\varepsilon_t$ enters the prediction with coefficient $1-\alpha_t$, so an extremely small gain retains LFU noise for many iterations even while it smooths the direct observations $Z_t$. Because $\widehat\Delta_t^{\mathrm{lag}}$ and $Z_t$ use the same $X_t$, their correlation affects variance but not the displayed conditional means.

These claims require the conditional distribution of $X_t$ to match the distribution defining $\mathcal I(\theta_t)$. Off-model data, uncorrected task drift, or temporal dependence beyond the conditioning state can introduce additional bias; reinforcement-learning experiments will need an appropriate conditional-Fisher or mixing interpretation.

### Effective sample size and forgetting

If the current-Fisher estimate will be used to construct future EWC factors, an effective-sample-size form uses a forgetting factor $\rho_{t-1}\in[0,1]$:

$$ \widetilde\Lambda_t=\rho_{t-1}n_{t-1}\left(\widehat{\mathcal I}_{t-1}+\widehat\Delta_t^{\mathrm{lag}}\right), $$

$$ \Lambda_t=\widetilde\Lambda_t+Z_t, \qquad n_t=\rho_{t-1}n_{t-1}+1, \qquad \widehat{\mathcal I}_t=\Lambda_t/n_t. $$

This corresponds to $\alpha_t=1/n_t$ for the displayed unit-weight observation update. Uniform long-run contraction requires the EMA gain to remain bounded away from zero; in the effective-count form, a fixed $\rho<1$ keeps $n_t$ bounded and produces such a gain. If $\rho=1$, then $\alpha_t=1/n_t\to0$, the update becomes an arithmetic average, and the constant-gain geometric bound above no longer applies. More general gains can decouple statistical smoothing from the effective count used by EWC, but then $n_t$ is a design quantity rather than a literal sample size.

### Autodiff implementation shape

For each sample, compute $g_t=\nabla L_t$ with a differentiable backward graph, form the scalar $g_t^Tu_{t-1}$, and differentiate that scalar once more to obtain $H_t$. The lagged LFU can then be stored as the two-column factorization

$$ U_t=[g_t,H_t], \qquad \widehat\Delta_t^{\mathrm{lag}}=U_t\begin{bmatrix}-(u_{t-1}^Tg_t)&1\\1&0\end{bmatrix}U_t^T. $$

The first gradient can be detached and reused by the optimizer when choosing $u_t$. This gives the lagged schedule a computational benefit: one observation and one gradient support both the previous LFU correction and the next learning update. In PyTorch, the extra HVP is primarily an additional reverse-mode pass; higher-order autodiff retains the forward graph and creates a graph for the first derivative, so peak memory can rise substantially even though activations are not simply duplicated and no dense Hessian is stored.

Because an LFU is a signed correction, a low-rank implementation should preserve signed factors rather than force every increment into a positive-semidefinite outer product. Positive semidefiniteness is a property to enforce on the resulting Fisher estimate, for example by damping, eigenvalue clipping in the maintained subspace, or a positive structured parameterization.


## Stochastic control for single-observation learning

The control model belongs to the original learning process and should remain separate from the auxiliary Fisher gain. Let $\pi_t\in[0,1]$ regulate adaptation to a local environmental displacement $d\theta_t$. If an experiment supports the conditional approximation

$$ \widehat\theta_{t+1}-\theta_t\approx\mathcal N\left(\pi_t d\theta_t,\frac{\pi_t}{n_t}\mathcal I^{-1}(\theta_t)\right), $$

then the one-step mean-squared error relative to $\theta_t+d\theta_t$ is

$$ \mathbb E\|\widehat\theta_{t+1}-(\theta_t+d\theta_t)\|^2\approx(1-\pi_t)^2\|d\theta_t\|^2+\frac{\pi_t}{n_t}\operatorname{tr}\mathcal I^{-1}(\theta_t). $$

The clipped local optimum is

$$ \pi_t^*=\operatorname{clip}_{[0,1]}\left(1-\frac{\operatorname{tr}[\mathcal I^{-1}(\theta_t)]}{2n_t\|d\theta_t\|^2+\varepsilon}\right), \qquad \varepsilon>0. $$

This result is conditional on the Gaussian update model; LFU neither requires nor proves that model. In practice $d\theta_t$ is unknown and must be estimated, so $\pi_t^*$ is best treated as a diagnostic or a component of a conservative controller.

A small-noise approximation for the original process may be written

$$ d\Theta_t^\varepsilon=\pi_t b(\Theta_t^\varepsilon)dt+\sqrt{\varepsilon\pi_t}\,\mathcal I^{-1/2}(\Theta_t^\varepsilon)dW_t. $$

The auxiliary process supplies the evolving estimate of $\mathcal I$ used by this model. This coupling is one-way at the level of the LFU derivation, but becomes two-way in an implementation when the Fisher estimate also shapes the EWC penalty and therefore changes future $u_t$.


## When local optimality demands forgetting

The one-step rule $\pi_t^*$ need not be a globally desirable learning policy. A large value says that the estimated system change is large relative to local statistical uncertainty. Locally, this favors rapid adaptation. Globally, tying that control directly to the auxiliary forgetting rate can erase accumulated curvature information.

For the online Fisher recursion

$$ \widehat{\mathcal I}_t=(1-\alpha_t)\left(\widehat{\mathcal I}_{t-1}+\widehat\Delta_t^{\mathrm{lag}}\right)+\alpha_t Z_t, $$

information from an earlier estimate is weighted after $m$ constant-gain steps by approximately $(1-\alpha)^m$. The learning control $\pi_t$, the Fisher gain $\alpha_t$, and the precision forgetting factor $\rho_t$ therefore have different meanings and should not be identified without an explicit design argument.

A practical controller can cap the adaptation rate,

$$ \pi_{\mathrm{used}}=\min(\widehat\pi_t^*,\pi_{\max}), $$

while independently setting a lower-bounded memory horizon through $\alpha_t$ or $\rho_t$. When the unconstrained diagnostic exceeds the cap, possible interventions include reducing the optimizer step, subdividing the move into several LFUs, increasing the batch used for curvature estimation, replaying selected observations, strengthening EWC, or freezing part of the trainable subspace.

Under this interpretation, a large $\widehat\pi_t^*$ is an overwhelm diagnostic: the locally optimal tracker would need to adapt too aggressively to preserve one-step accuracy. It is evidence that the system may be outside the regime where a compressed local quadratic and a first-order Fisher update are reliable.


## Citations

[1] Y. Zheng, Y. Zhang, J. van de Weijer, G. M. van de Ven, S. Du, X. Zhang, and Z. Tian, [*Revisiting Weight Regularization for Low-Rank Continual Learning*](https://arxiv.org/abs/2602.17559), arXiv:2602.17559, 2026.

[2] S. Amari and H. Nagaoka, *Methods of Information Geometry*, American Mathematical Society, 2000.

[3] J. Kirkpatrick et al., "Overcoming catastrophic forgetting in neural networks," *Proceedings of the National Academy of Sciences*, 114(13), 3521-3526, 2017.

[4] B. A. Pearlmutter, "Fast Exact Multiplication by the Hessian," *Neural Computation*, 6(1), 147-160, 1994.
